# Can a mathematical route loop back to an earlier date?

**The idea in everyday language:** Imagine two special regions of space-time. In a published mathematical example, a traveler passes through both and can return to an earlier event while their own clock keeps ticking forward. The equations describe a possible shape of space-time; they do not show how to build those regions or a ship that can travel through them.

This calculator checks one rule for when the loop can close. It also tries a much simpler way to estimate where an ancient Earth event would appear in the traveler's starting frame. That second calculation gives a very large distance, so it is a scale warning, not a route to Bethlehem.

**What the numbers mean:** they follow from the chosen equations and inputs. They do not prove that A.R.G.O. can make the trip. The Star story also leaves the exact date and viewing direction unknown.

## The calculation in detail

This chapter reproduces the symmetric closure benchmark discussed by Shoshany and Snodgrass and evaluates a separate flat 1+1 endpoint relation. The endpoint equation is a kinematic extension, not the paper's compact-support metric retargeted to Bethlehem.

The endpoint frame $S$ is the departure-comoving inertial frame used by the published construction: the traveler starts at rest there. A consistent target transform must include Earth's orbital velocity in A.R.G.O.'s initial state. In a one-axis illustration, assume the Sun's Galactic velocity and Earth's orbital velocity are collinear, transform the historical Earth event from a Galactic frame into $S$, and carry projected bounds for orbital phase, ground-site rotation, and Galactic curvature. This is a scenario calculation, not an ephemeris reconstruction.

For the paper's illustrative drive parameters, the transformed ancient Earth event is about $0.202$ light-years away in the chosen projection. The flat endpoint algebra still asks for a first leg of about $16{,}558$ years and $38{,}963$ light-years. That galactic-scale result diagnoses the endpoint extension as unsuitable for navigation; it is not a Bethlehem route or a solution of the paper's full metric.

The Star narrative supplies no exact epoch, range, or observer worldline. Treat 6 BCE only as a scenario label and explore a family of endpoint states.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from IPython.display import HTML, display

# Parameters of the paper's explicit smooth-profile example.
U0, alpha0, omega0 = 0.75, 10.0, 0.95
lookback_years = 2031.0  # Illustrative scale only; the source narrative gives no exact year.
solar_galactic_speed_km_s = 230.0
earth_orbital_speed_km_s = 29.78
earth_rotation_speed_bound_km_s = 0.465  # Equatorial upper bound for a surface site.
galactocentric_radius_kpc = 8.2
earth_orbit_offset_bound_au = 2.0
seconds_per_year = 365.25 * 24 * 3600
meters_per_lightyear = 9.4607304725808e15
meters_per_au = 149_597_870_700.0
meters_per_kpc = 3.085677581491367e19
c_km_s = 299_792.458

# One-axis Galactic-frame to departure-rest-frame transform.
beta_sun = solar_galactic_speed_km_s / c_km_s
beta_earth_orbit = earth_orbital_speed_km_s / c_km_s
beta_departure = (beta_sun + beta_earth_orbit) / (1 + beta_sun * beta_earth_orbit)
gamma_departure = 1 / np.sqrt(1 - beta_departure**2)

def endpoint_transform(lookback):
    dt_galactic = -lookback
    dz_galactic = -beta_sun * lookback  # Shared linear Galactic drift; same solar-orbit phase assumed centrally.
    dt_departure = gamma_departure * (dt_galactic - beta_departure * dz_galactic)
    dz_departure = gamma_departure * (dz_galactic - beta_departure * dt_galactic)
    return dt_departure, dz_departure

def endpoint_coefficients(U, V):
    return (2 * V - U * (1 + V**2)) / (V - U), (V * U - 1) / (V - U)

V0 = U0 * (1 + omega0 * (alpha0 - 1) / 4)
A0, B0 = endpoint_coefficients(U0, V0)
t_target, z_target_ly = endpoint_transform(lookback_years)
galactic_acceleration = (solar_galactic_speed_km_s * 1000)**2 / (galactocentric_radius_kpc * meters_per_kpc)
galactic_curvature_bound_ly = 0.5 * galactic_acceleration * (lookback_years * seconds_per_year)**2 / meters_per_lightyear
earth_orbit_phase_bound_ly = earth_orbit_offset_bound_au * meters_per_au / meters_per_lightyear
earth_rotation_drift_bound_ly = earth_rotation_speed_bound_km_s / c_km_s * lookback_years
target_offset_bound_ly = galactic_curvature_bound_ly + earth_orbit_phase_bound_ly + earth_rotation_drift_bound_ly
z_target_min_ly, z_target_max_ly = z_target_ly - target_offset_bound_ly, z_target_ly + target_offset_bound_ly
T_first = (t_target - B0 * z_target_ly) / A0
D_first = V0 * T_first
T_orbit_bounds = sorted([(t_target - B0 * z_target_min_ly) / A0,
                         (t_target - B0 * z_target_max_ly) / A0])
print(f"V = {V0:.6f} c")
print(f"Ucrit = {2*V0/(1+V0**2):.8f}; chosen U = {U0:.3f}")
print(f"Tfinish/T2 = {A0:.8f}")
print(f"Departure speed in Galactic frame (collinear assumption) = {beta_departure*c_km_s:.3f} km/s")
print(f"Transformed target event: t = {t_target:,.6f} y, z = {z_target_ly:.6f} ly")
print(f"Projected target-position sensitivity = ±{target_offset_bound_ly:.6f} ly")
print(f"Flat endpoint estimate: T2 = {T_first:,.2f} y, first-leg separation = {D_first:,.2f} ly")
print(f"Symmetric-family lower bound for this target endpoint: D > {abs(t_target)-abs(z_target_ly):,.2f} ly")
print(f"First-leg time range under projected offset bounds: {T_orbit_bounds[0]:,.3f}–{T_orbit_bounds[1]:,.3f} y")
closure_speed_fraction_c = z_target_ly / t_target
closure_proper_time_years = np.sqrt(t_target**2 - z_target_ly**2)
print(f"Straight target-to-departure segment: timelike={abs(z_target_ly) < abs(t_target)}, velocity={closure_speed_fraction_c*c_km_s:.2f} km/s, proper time={closure_proper_time_years:,.4f} y")
print("Endpoint check only: tangent matching to the return-bubble exit is not solved.")

## Closed-loop threshold versus average bubble speed

In [ ]:
V_grid = np.linspace(1.001, 8.0, 500)
Ucrit_grid = 2 * V_grid / (1 + V_grid**2)
fig = go.Figure()
fig.add_trace(go.Scatter(x=V_grid, y=Ucrit_grid, name="minimum frame speed", mode="lines"))
fig.add_hline(y=U0, line_dash="dash", annotation_text=f"example U={U0}")
fig.add_vline(x=V0, line_dash="dot", annotation_text=f"example V={V0:.3f}")
fig.update_layout(title="Symmetric CTC threshold", xaxis_title="Average bubble separation speed V / c", yaxis_title="Required relative frame speed Ucrit / c", yaxis_range=[0,1], template="plotly_white")
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))

## Flat endpoint extension: a scale diagnostic, not a mission trajectory

The plotted target event is transformed from a one-axis Galactic frame to the departure-comoving frame. The central scenario assumes collinear solar Galactic and Earth-orbital velocities and equal heliocentric orbital phase at the two epochs. The shaded interval adds projected Earth orbital-phase, surface-rotation, and Galactic-curvature scales; it is not a statistical error bar or complete uncertainty budget. This calculation does not reconstruct Earth's state or locate a Bethlehem observer. It does not integrate a Galactic potential, retarget the published finite-support metric, or solve the full geodesic. The thousands-of-light-years output shows that this endpoint extension is outside a credible local navigation regime.

**Try it:** Move the slider to choose a lookback time. The marker follows the model's central distance estimate. The shaded band is the same projected sensitivity scale described above. The year is a scenario input, not a date established by the Star narrative.

For the symmetric two-leg algebra, a target endpoint timelike-separated from departure has the necessary bound $D>|t_C|-|z_C|$. The dashed lower-bound curve applies to the central one-axis target transform; it is not a bound on every point of the published closed geodesic. The 100 AU line is a solar-system-scale reference. The actual finite-support metric and tangent-matched route are not solved here.

In [ ]:
lookbacks = np.linspace(1, 3000, 480)
t_targets, z_targets = endpoint_transform(lookbacks)
galactic_curvature_bounds = 0.5 * galactic_acceleration * (lookbacks * seconds_per_year)**2 / meters_per_lightyear
earth_orbit_bounds = np.full_like(lookbacks, earth_orbit_phase_bound_ly)
earth_rotation_bounds = earth_rotation_speed_bound_km_s / c_km_s * lookbacks
target_bounds = galactic_curvature_bounds + earth_orbit_bounds + earth_rotation_bounds
T_center = (t_targets - B0 * z_targets) / A0
D_center = V0 * T_center
D_bound_a = V0 * ((t_targets - B0 * (z_targets - target_bounds)) / A0)
D_bound_b = V0 * ((t_targets - B0 * (z_targets + target_bounds)) / A0)
D_low = np.minimum(D_bound_a, D_bound_b)
D_high = np.maximum(D_bound_a, D_bound_b)
fig = go.Figure()
fig.add_trace(go.Scatter(x=lookbacks, y=D_low, mode="lines", line=dict(width=0), showlegend=False, hoverinfo="skip"))
fig.add_trace(go.Scatter(x=lookbacks, y=D_high, mode="lines", line=dict(width=0), fill="tonexty", fillcolor="rgba(31,119,180,0.18)", name="projected target-offset sensitivity"))
fig.add_trace(go.Scatter(x=lookbacks, y=D_center, mode="lines", name="departure-frame central estimate"))
D_floor = np.abs(t_targets) - np.abs(z_targets)
fig.add_trace(go.Scatter(x=lookbacks, y=D_floor, mode="lines", line=dict(dash="dash"), name="symmetric-family lower bound"))
slider_years = np.unique(np.r_[np.arange(1, 2501, 50), lookback_years]).astype(int)
slider_distances = np.interp(slider_years, lookbacks, D_center)
fig.add_trace(go.Scatter(x=[lookback_years], y=[D_first], mode="markers", marker=dict(size=12, color="#c23b22"), name="selected lookback", hovertemplate="Lookback: %{x:,} years<br>Central estimate: %{y:,.0f} ly<extra></extra>"))
steps = [dict(method="update", label=f"{year:,}", args=[{"x": [[year]], "y": [[distance]]}, {"title": f"Flat-model scale at {year:,} years lookback"}, [4]]) for year, distance in zip(slider_years, slider_distances)]
fig.update_layout(title=f"Flat-model scale at {lookback_years:,} years lookback", xaxis_title="Lookback in Galactic frame (years; scenario input)", yaxis_title="First-leg separation (light-years)", yaxis_type="log", template="plotly_white", shapes=[dict(type="line", xref="paper", x0=0, x1=1, y0=100*meters_per_au/meters_per_lightyear, y1=100*meters_per_au/meters_per_lightyear, line=dict(color="gray", dash="dot"))], annotations=[dict(x=1, xref="paper", y=100*meters_per_au/meters_per_lightyear, text="100 AU reference", showarrow=False, yshift=10)], sliders=[dict(active=int(np.argmin(abs(slider_years-lookback_years))), currentvalue=dict(prefix="Target lookback: "), pad=dict(t=35), steps=steps)])
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))

## Retargeting gap

The one-axis transform repairs the most direct frame inconsistency but does not produce a navigable target state. Galactic and heliocentric velocities are not actually collinear; orbital phases, observer location, Earth's rotation, non-circular Galactic motion, and gravitational fields need vector ephemerides in declared frames. A precision route also needs the retargeted metric and a full closed timelike geodesic intersecting both encounter and departure events. Changing this flat endpoint does not move the finite-support profiles, recompute the lapse, or solve the geodesic. See the [candidate review](../temporal-drive-candidate.md).

## A continuous loop can reach the date—and still collide

The previous plot treats the target as the endpoint of a simplified flat-space extension. Here we use the explicit smooth two-bubble centerline from the published construction and scale its full loop so that an interior point reaches the same central 6 BCE scenario event. The traveler remains on one continuous, future-directed timelike curve; it does not jump between dates.

The result is a failure case, not a route: in this one-axis model the chosen target is **Earth's center**, and the curve intersects it at about 1,300 km/s. A point coincidence with Earth's center is a collision. The calculation still omits 3D Earth and site motion, gravity, the finite ship and bubble wall, and photons traveling from A.R.G.O. to a witness. A safe appearance requires a full 3D world-tube clearance calculation and a solved light path.

**Try it:** Change the frame speed, profile parameters, or scenario event. The plot shows the entire loop and marks the target crossing. The notebook also prints the target position residual so numerical precision is visible. The arXiv paper establishes the analytic curve as a geodesic of its prescribed metric; this calculator evaluates that centerline and does not establish a realizable drive. [Source paper](https://arxiv.org/abs/2309.10072).


In [ ]:
from pathlib import Path
import sys
repo_root = Path.cwd()
if not (repo_root / "scripts").is_dir():
    repo_root = next(parent for parent in Path.cwd().parents if (parent / "scripts").is_dir())
sys.path.insert(0, str(repo_root / "scripts"))
from solve_two_bubble_encounter import calculate_encounter, sample_worldline

encounter = calculate_encounter(
    target_time_years=float(t_target), target_z_lightyears=float(z_target_ly),
    frame_speed=U0, alpha=alpha0, omega=omega0,
)
path = sample_worldline(encounter, count=1800)
print(f"Interior target event: ({encounter['target_event_time_S_years']:.6f} y, "
      f"{encounter['target_event_z_S_lightyears']:.9f} ly)")
print(f"Position residual: {encounter['target_coordinate_residual_lightyears']:.3e} ly")
print(f"Return-leg transition fraction: {encounter['return_transition_fraction']:.6f}")
print(f"First-leg separation: {encounter['first_leg_separation_lightyears']:,.1f} ly")
print(f"Traveler proper time to target: {encounter['traveler_proper_time_at_target_years']:,.1f} y")
print(f"Coordinate speed at crossing: {encounter['S_coordinate_velocity_z_fraction_c']*c_km_s:,.0f} km/s")
print("Safety result: Earth-center intersection — collision in this model.")

fig_loop = go.Figure()
fig_loop.add_trace(go.Scatter(x=path["outbound_t"], y=path["outbound_z"], mode="lines", name="First bubble leg"))
fig_loop.add_trace(go.Scatter(x=path["return_t"], y=path["return_z"], mode="lines", name="Second bubble leg"))
fig_loop.add_trace(go.Scatter(x=path["closure_t"], y=path["closure_z"], mode="lines", name="Continuous closing segment", line=dict(dash="dot")))
fig_loop.add_trace(go.Scatter(x=[t_target], y=[z_target_ly], mode="markers", name="6 BCE Earth-center event (collision)", marker=dict(size=12, color="#ef4444", symbol="x")))
fig_loop.update_layout(title="Explicit two-bubble loop through the modeled Earth-center event", xaxis_title="Departure-frame coordinate time (years)", yaxis_title="Displacement along modeled axis (light-years)", hovermode="closest", legend=dict(orientation="h", y=-0.25), height=560)
fig_loop.update_xaxes(showgrid=True, zeroline=True)
fig_loop.update_yaxes(showgrid=True, zeroline=True)
fig_loop.show()
